In [1]:
SEED = 42
VAL_FRAC = 0.10

EPOCHS = 8
BATCH = 1024
LR = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2
HIDDEN = 2048
DROPOUT = 0.2

ENSEMBLE_SEEDS = [42, 1337, 2026]

KNN_K = 60

OUT_PATH = "/kaggle/working/submission.tsv"
MAX_TOTAL_TERMS = 1500
TOP_PER_ASPECT = {"BP": 500, "MF": 500, "CC": 500} 
TOPK_MLP = 300
MIN_WRITE_SCORE = 1e-6 

In [2]:
import os, re, json, math
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.amp import autocast, GradScaler

In [3]:
def read_fasta_to_df(path: str) -> pd.DataFrame:
    ids, seqs = [], []
    cur_id, cur_seq = None, []
    with open(path, "r", encoding="utf-8") as f:
        for raw in f:
            line = raw.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur_id is not None:
                    ids.append(cur_id); seqs.append("".join(cur_seq))
                cur_id = line[1:].split()[0]
                cur_seq = []
            else:
                cur_seq.append(line)
    if cur_id is not None:
        ids.append(cur_id); seqs.append("".join(cur_seq))
    return pd.DataFrame({"protein_id": ids, "sequence": seqs})

def normalize_protein_id(x: str) -> str:
    x = str(x).strip().split()[0]
    if "|" in x:
        parts = x.split("|")
        x = parts[1] if len(parts) >= 2 else parts[-1]
    if "." in x:
        left, right = x.rsplit(".", 1)
        if right.isdigit():
            x = left
    return x

def l2norm(X, eps=1e-12):
    X = np.asarray(X, dtype=np.float32)
    n = np.linalg.norm(X, axis=1, keepdims=True)
    return X / (n + eps)

def make_split_embeddings(pids, X_all, id_to_row):
    rows, kept = [], []
    for pid in pids:
        r = id_to_row.get(pid, None)
        if r is None:
            continue
        rows.append(r); kept.append(pid)
    X = np.asarray(X_all[rows], dtype=np.float32)
    return kept, X

def fmt_3sig(x: float):
    x = float(x)
    if not (x > 0.0):
        return None
    if x > 1.0:
        x = 1.0
    s = f"{x:.3g}"
    if s == "0":
        return None
    return s

In [4]:
COMP_DIR = "/kaggle/input/cafa-6-protein-function-prediction"
TRAIN_DIR = os.path.join(COMP_DIR, "Train")
TEST_DIR  = os.path.join(COMP_DIR, "Test")

TRAIN_TERMS = os.path.join(TRAIN_DIR, "train_terms.tsv")
TRAIN_FASTA = os.path.join(TRAIN_DIR, "train_sequences.fasta")
TEST_FASTA  = os.path.join(TEST_DIR,  "testsuperset.fasta")
IA_PATH     = os.path.join(COMP_DIR, "IA.tsv")

EMB_DIR = "/kaggle/input/cafa6-protein-embeddings-esm2"

In [5]:
train_seq = read_fasta_to_df(TRAIN_FASTA)
test_seq  = read_fasta_to_df(TEST_FASTA)

train_seq["protein_id"] = train_seq["protein_id"].map(normalize_protein_id)
test_seq["protein_id"]  = test_seq["protein_id"].map(normalize_protein_id)

train_terms = pd.read_csv(TRAIN_TERMS, sep="\t")
train_terms = train_terms.rename(columns={"EntryID":"protein_id","term":"go_id"})
train_terms["protein_id"] = train_terms["protein_id"].map(normalize_protein_id)

train_terms["aspect"] = train_terms["aspect"].astype(str).str.strip().str.upper()
aspect_map = {"P":"BP","F":"MF","C":"CC","BP":"BP","MF":"MF","CC":"CC"}
train_terms["asp_norm"] = train_terms["aspect"].map(aspect_map)

print(train_seq.shape, test_seq.shape, train_terms.shape)
print(train_terms["asp_norm"].value_counts())

(82404, 2) (224309, 2) (537027, 4)
asp_norm
BP    250805
CC    157770
MF    128452
Name: count, dtype: int64


In [6]:
rng = np.random.default_rng(SEED)

seq_to_rep = train_seq.groupby("sequence")["protein_id"].min().rename("rep_id").reset_index()
train_rep = train_seq.merge(seq_to_rep, on="sequence", how="left")

rep_ids = train_rep["rep_id"].drop_duplicates().values
rng.shuffle(rep_ids)

n_val = int(len(rep_ids) * VAL_FRAC)
val_rep = set(rep_ids[:n_val])
train_rep_set = set(rep_ids[n_val:])

val_ids   = train_rep.loc[train_rep["rep_id"].isin(val_rep), "protein_id"].unique().tolist()
train_ids = train_rep.loc[train_rep["rep_id"].isin(train_rep_set), "protein_id"].unique().tolist()

print("train:", len(train_ids), "val:", len(val_ids), "overlap:", len(set(train_ids)&set(val_ids)))

train: 74150 val: 8254 overlap: 0


In [7]:
emb_path = os.path.join(EMB_DIR, "protein_embeddings.npy")
id_path  = os.path.join(EMB_DIR, "protein_ids.csv")

X_all = np.load(emb_path, mmap_mode="r")
ids_all = pd.read_csv(id_path, header=None)[0].astype(str).map(normalize_protein_id).tolist()

print("X_all:", X_all.shape, "ids:", len(ids_all))

n = min(len(ids_all), X_all.shape[0])
ids_all = ids_all[:n]
X_all = X_all[:n]

id_to_row = {pid:i for i,pid in enumerate(ids_all)}

train_pids_aligned, X_train = make_split_embeddings(train_ids, X_all, id_to_row)
val_pids_aligned,   X_val   = make_split_embeddings(val_ids,   X_all, id_to_row)
test_pids_aligned,  X_test  = make_split_embeddings(test_seq["protein_id"].tolist(), X_all, id_to_row)

missing_ids = list(set(test_seq["protein_id"]) - set(test_pids_aligned))
print("aligned train/val/test:", len(train_pids_aligned), len(val_pids_aligned), len(test_pids_aligned))
print("missing embeddings:", missing_ids[:10], "count:", len(missing_ids))

X_train_n = l2norm(X_train)
X_val_n   = l2norm(X_val)
X_test_n  = l2norm(X_test)

X_all: (287001, 1280) ids: 287002
aligned train/val/test: 74150 8254 224308
missing embeddings: ['A4QQE0'] count: 1


In [8]:
ASPECTS = ["BP","MF","CC"]

terms_train_split = train_terms[train_terms["protein_id"].isin(train_ids)].copy()

go_vocab = {}
go2i = {}

for asp in ASPECTS:
    vocab = sorted(terms_train_split.loc[terms_train_split["asp_norm"]==asp, "go_id"].unique())
    go_vocab[asp] = vocab
    go2i[asp] = {go:i for i,go in enumerate(vocab)}
    print(asp, "labels:", len(vocab))

def pid2labels_for_aspect(pids_set, asp):
    df = train_terms[(train_terms["protein_id"].isin(pids_set)) & (train_terms["asp_norm"]==asp)][["protein_id","go_id"]].copy()
    df = df[df["go_id"].isin(go2i[asp])]
    df["y"] = df["go_id"].map(go2i[asp]).astype(int)
    return df.groupby("protein_id")["y"].apply(list).to_dict()

train_pid2labels = {asp: pid2labels_for_aspect(set(train_pids_aligned), asp) for asp in ASPECTS}
val_pid2labels   = {asp: pid2labels_for_aspect(set(val_pids_aligned),   asp) for asp in ASPECTS}

top_fallback = {}
for asp in ASPECTS:
    s = train_terms[train_terms["asp_norm"]==asp]["go_id"].value_counts()
    top_fallback[asp] = s.head(200).index.tolist()
    print(asp, "fallback terms:", len(top_fallback[asp]), "top1:", top_fallback[asp][0])

BP labels: 16525
MF labels: 6442
CC labels: 2609
BP fallback terms: 200 top1: GO:0045944
MF fallback terms: 200 top1: GO:0005515
CC fallback terms: 200 top1: GO:0005634


In [9]:
class EmbDataset(Dataset):
    def __init__(self, X, pids, pid2labels):
        self.X = X
        self.pids = pids
        self.pid2labels = pid2labels
    def __len__(self): return len(self.pids)
    def __getitem__(self, i):
        pid = self.pids[i]
        labs = self.pid2labels.get(pid, [])
        return self.X[i], labs

def collate_emb(batch, n_labels):
    xs, labs_list = zip(*batch)
    Xb = torch.tensor(np.stack(xs), dtype=torch.float32)
    Yb = torch.zeros((len(xs), n_labels), dtype=torch.float32)
    for i,labs in enumerate(labs_list):
        if labs:
            Yb[i, labs] = 1.0
    return Xb, Yb

class ASLStable(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=2.0):
        super().__init__()
        self.gp = gamma_pos
        self.gn = gamma_neg
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        p = torch.sigmoid(logits)
        pt = p*targets + (1-p)*(1-targets)
        w = (1-pt).pow(self.gp*targets + self.gn*(1-targets))
        return (w*bce).mean()

class MLP(nn.Module):
    def __init__(self, d_in, d_hid, n_out, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_in, d_hid),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hid, n_out),
        )
    def forward(self, x): return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
scaler = GradScaler("cuda", enabled=(device.type=="cuda"))
D = X_train.shape[1]
print("device:", device, "D:", D)

device: cuda D: 1280


In [10]:
def train_one(asp, seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(seed)

    nlab = len(go_vocab[asp])
    train_ds = EmbDataset(X_train, train_pids_aligned, train_pid2labels[asp])
    val_ds   = EmbDataset(X_val,   val_pids_aligned,   val_pid2labels[asp])

    train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=True,
                              collate_fn=lambda b, nlab=nlab: collate_emb(b, nlab))
    val_loader = DataLoader(val_ds, batch_size=BATCH, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True,
                            collate_fn=lambda b, nlab=nlab: collate_emb(b, nlab))

    model = MLP(D, HIDDEN, nlab, dropout=DROPOUT).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    loss_fn = ASLStable(gamma_pos=0.0, gamma_neg=2.0)

    for ep in range(1, EPOCHS+1):
        model.train()
        for Xb, Yb in train_loader:
            Xb = Xb.to(device, non_blocking=True)
            Yb = Yb.to(device, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with autocast("cuda", enabled=(device.type=="cuda")):
                logits = model(Xb)
                loss = loss_fn(logits, Yb)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        print(f"{asp} seed {seed} epoch {ep} done")
    return model

models = {asp: [] for asp in ASPECTS}
for asp in ASPECTS:
    for s in ENSEMBLE_SEEDS:
        models[asp].append(train_one(asp, s))

BP seed 42 epoch 1 done
BP seed 42 epoch 2 done
BP seed 42 epoch 3 done
BP seed 42 epoch 4 done
BP seed 42 epoch 5 done
BP seed 42 epoch 6 done
BP seed 42 epoch 7 done
BP seed 42 epoch 8 done
BP seed 1337 epoch 1 done
BP seed 1337 epoch 2 done
BP seed 1337 epoch 3 done
BP seed 1337 epoch 4 done
BP seed 1337 epoch 5 done
BP seed 1337 epoch 6 done
BP seed 1337 epoch 7 done
BP seed 1337 epoch 8 done
BP seed 2026 epoch 1 done
BP seed 2026 epoch 2 done
BP seed 2026 epoch 3 done
BP seed 2026 epoch 4 done
BP seed 2026 epoch 5 done
BP seed 2026 epoch 6 done
BP seed 2026 epoch 7 done
BP seed 2026 epoch 8 done
MF seed 42 epoch 1 done
MF seed 42 epoch 2 done
MF seed 42 epoch 3 done
MF seed 42 epoch 4 done
MF seed 42 epoch 5 done
MF seed 42 epoch 6 done
MF seed 42 epoch 7 done
MF seed 42 epoch 8 done
MF seed 1337 epoch 1 done
MF seed 1337 epoch 2 done
MF seed 1337 epoch 3 done
MF seed 1337 epoch 4 done
MF seed 1337 epoch 5 done
MF seed 1337 epoch 6 done
MF seed 1337 epoch 7 done
MF seed 1337 epoch

In [11]:
ia_df = pd.read_csv(IA_PATH, sep="\t")
ia_df.columns = [c.lower() for c in ia_df.columns]
term_col = "term" if "term" in ia_df.columns else ia_df.columns[0]
ia_col   = "ia"   if "ia"   in ia_df.columns else ia_df.columns[1]
IA = dict(zip(ia_df[term_col].astype(str), ia_df[ia_col].astype(float)))

def weighted_fmax(val_pids, pid2labels, i2go, Y_score, thresholds=np.linspace(0.01, 0.99, 99)):
    truth = {}
    for pid in val_pids:
        labs = pid2labels.get(pid, [])
        truth[pid] = set(i2go[j] for j in labs)

    w_label = np.array([IA.get(go, 0.0) for go in i2go], dtype=np.float32)

    best = (-1.0, None, None, None)  # f, thr, wP, wR
    K = min(200, Y_score.shape[1])
    top_idx = np.argpartition(-Y_score, K-1, axis=1)[:, :K]

    for thr in thresholds:
        wp_sum = 0.0
        wr_sum = 0.0
        wtp_sum = 0.0

        for i, pid in enumerate(val_pids):
            idx = top_idx[i]
            sc  = Y_score[i, idx]
            keep = idx[sc >= thr]

            if keep.size == 0:
                pred_set = set()
                pred_w = 0.0
            else:
                pred_set = set(i2go[j] for j in keep.tolist())
                pred_w = float(w_label[keep].sum())

            true_set = truth[pid]
            true_w = float(sum(IA.get(go, 0.0) for go in true_set))

            if pred_set:
                inter = pred_set & true_set
                tp_w = float(sum(IA.get(go, 0.0) for go in inter))
            else:
                tp_w = 0.0

            wp_sum += pred_w
            wr_sum += true_w
            wtp_sum += tp_w

        wP = wtp_sum / (wp_sum + 1e-9)
        wR = wtp_sum / (wr_sum + 1e-9)
        wF = 2*wP*wR / (wP + wR + 1e-9)

        if wF > best[0]:
            best = (float(wF), float(thr), float(wP), float(wR))

    return best

@torch.no_grad()
def ensemble_probs(asp, Xb):
    ms = models[asp]
    lg = None
    for m in ms:
        m = m.to(device).eval()
        xb = torch.from_numpy(Xb).to(device)
        z = m(xb)
        lg = z if lg is None else (lg + z)
    lg = lg / float(len(ms))
    pr = torch.sigmoid(lg).float().cpu().numpy()
    if np.isnan(pr).any():
        raise ValueError(f"{asp}: NaNs in probs")
    return pr

thr_aspect = {}
for asp in ASPECTS:
    nlab = len(go_vocab[asp])
    i2go = go_vocab[asp]
    outs = []
    bs = 2048
    for i0 in range(0, X_val.shape[0], bs):
        outs.append(ensemble_probs(asp, X_val[i0:i0+bs]))
    Y = np.vstack(outs).astype(np.float32)
    bestF, thr, wP, wR = weighted_fmax(val_pids_aligned, val_pid2labels[asp], i2go, Y)
    thr_aspect[asp] = thr
    print(asp, "best weightedF:", bestF, "thr:", thr, "wP:", wP, "wR:", wR)

thr_aspect

BP best weightedF: 0.011655400715219068 thr: 0.12 wP: 0.006886695083845839 wR: 0.03789782917353371
MF best weightedF: 0.07290264264133717 thr: 0.2 wP: 0.06665817519174713 wR: 0.08043798964914711
CC best weightedF: 0.15594374875703892 thr: 0.3 wP: 0.15361885864933394 wR: 0.15834009170634603


{'BP': 0.12, 'MF': 0.2, 'CC': 0.3}

In [12]:
!pip -q install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 69.2 MB/s eta 0:00:00:00:0100:01


In [13]:
import faiss
print("faiss version:", faiss.__version__)

faiss version: 1.13.2


In [14]:
import faiss

index = faiss.IndexFlatIP(D)
index.add(X_train_n)

train_go_sets = {}
for asp in ASPECTS:
    i2go = go_vocab[asp]
    d = {}
    for pid in train_pids_aligned:
        labs = train_pid2labels[asp].get(pid, [])
        d[pid] = set(i2go[j] for j in labs)
    train_go_sets[asp] = d

def knn_scores_batch(Xb_n, asp, K=60):
    sims, nbrs = index.search(Xb_n, K)
    out = []
    for i in range(Xb_n.shape[0]):
        scores = {}
        for s, ridx in zip(sims[i], nbrs[i]):
            if ridx < 0:
                continue
            pid_n = train_pids_aligned[int(ridx)]
            w = float(max(s, 0.0))
            if w <= 0:
                continue
            for go in train_go_sets[asp][pid_n]:
                scores[go] = scores.get(go, 0.0) + w
        if scores:
            mx = max(scores.values())
            for go in list(scores.keys()):
                scores[go] = min(1.0, scores[go] / (mx + 1e-9))
        out.append(scores)
    return out

def dense_from_knn_dicts(knn_dicts, asp):
    i2go = go_vocab[asp]
    go2i_local = {go:i for i,go in enumerate(i2go)}
    Y = np.zeros((len(knn_dicts), len(i2go)), dtype=np.float32)
    for i,d in enumerate(knn_dicts):
        for go,s in d.items():
            j = go2i_local.get(go, None)
            if j is not None:
                if s > Y[i,j]:
                    Y[i,j] = float(s)
    return Y

alpha_grid = [0.2,0.4,0.6,0.8]
alpha_aspect = {}

for asp in ASPECTS:
    outs = []
    bs = 2048
    for i0 in range(0, X_val.shape[0], bs):
        outs.append(ensemble_probs(asp, X_val[i0:i0+bs]))
    Y_mlp = np.vstack(outs).astype(np.float32)

    knn_dicts = knn_scores_batch(X_val_n, asp, K=KNN_K)
    Y_knn = dense_from_knn_dicts(knn_dicts, asp)

    bestF = -1
    bestA = None
    bestThr = None
    for a in alpha_grid:
        Y_blend = (a * Y_mlp + (1-a) * Y_knn).astype(np.float32)
        bestF2, thr2, wP2, wR2 = weighted_fmax(val_pids_aligned, val_pid2labels[asp], go_vocab[asp], Y_blend)
        if bestF2 > bestF:
            bestF = bestF2
            bestA = a
            bestThr = thr2

    alpha_aspect[asp] = bestA
    thr_aspect[asp] = bestThr
    print(asp, "best weightedF:", bestF, "alpha:", bestA, "thr:", bestThr)

alpha_aspect, thr_aspect

BP best weightedF: 0.007592760089137701 alpha: 0.8 thr: 0.18000000000000002
MF best weightedF: 0.07138011251344796 alpha: 0.8 thr: 0.17
CC best weightedF: 0.15282027254004682 alpha: 0.8 thr: 0.35000000000000003


({'BP': 0.8, 'MF': 0.8, 'CC': 0.8},
 {'BP': 0.18000000000000002, 'MF': 0.17, 'CC': 0.35000000000000003})

In [ ]:
@torch.no_grad()
def ensemble_topk(asp, Xb, topk=300):
    ms = models[asp]
    xb = torch.from_numpy(Xb.astype(np.float32)).to(device)
    lg = None
    for m in ms:
        m = m.to(device).eval()
        z = m(xb)
        lg = z if lg is None else (lg + z)
    lg = lg / float(len(ms))
    pr = torch.sigmoid(lg).float()
    k = min(topk, pr.shape[1])
    sc, idx = torch.topk(pr, k=k, dim=1)
    return idx.cpu().numpy(), sc.cpu().numpy()

written = 0
with open(OUT_PATH, "w", encoding="utf-8") as f:
    for i0 in range(0, X_test.shape[0], 512):
        Xb = X_test[i0:i0+512]
        Xb_n = X_test_n[i0:i0+512]
        B = Xb.shape[0]

        per_asp = {asp:[None]*B for asp in ASPECTS}

        for asp in ASPECTS:
            a = float(alpha_aspect[asp])
            thr = float(thr_aspect[asp])
            i2go = go_vocab[asp]

            knn_dicts = knn_scores_batch(Xb_n, asp, K=KNN_K)

            idx, sc = ensemble_topk(asp, Xb, topk=TOPK_MLP)

            out = []
            for r in range(B):
                d = {}
                for go,s in knn_dicts[r].items():
                    d[go] = (1-a) * float(s)
                for j,s in zip(idx[r], sc[r]):
                    go = i2go[int(j)]
                    d[go] = d.get(go, 0.0) + a * float(s)

                items = [(go,s) for go,s in d.items() if s >= thr and s > MIN_WRITE_SCORE]
                items.sort(key=lambda x: x[1] * (1.0 + IA.get(x[0], 0.0)), reverse=True)
                items = items[:TOP_PER_ASPECT[asp]]
                out.append(items)
            per_asp[asp] = out

        for r in range(B):
            pid = test_pids_aligned[i0 + r]
            combined = []
            for asp in ASPECTS:
                combined.extend(per_asp[asp][r])
            combined.sort(key=lambda x: x[1] * (1.0 + IA.get(x[0], 0.0)), reverse=True)
            combined = combined[:MAX_TOTAL_TERMS]

            for go,s in combined:
                s3 = fmt_3sig(s)
                if s3 is None:
                    continue
                f.write(f"{pid}\t{go}\t{s3}\n")
                written += 1

    for pid in missing_ids:
        for asp in ASPECTS:
            for go in top_fallback[asp][:50]:
                f.write(f"{pid}\t{go}\t0.1\n")
                written += 1

print("Wrote:", OUT_PATH, "rows:", written)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ASPECTS = ["BP","MF","CC"]

for asp in ASPECTS:
    if isinstance(models[asp], (list, tuple)):
        models[asp] = [m.to(device).eval() for m in models[asp]]
    else:
        models[asp] = models[asp].to(device).eval()

In [ ]:
import torch

@torch.no_grad()
def predict_topk_ensemble(models_for_asp, xb, k):
    if isinstance(models_for_asp, (list, tuple)):
        lg_sum = None
        for m in models_for_asp:
            lg = m(xb)  # logits
            lg_sum = lg if lg_sum is None else (lg_sum + lg)
        lg = lg_sum / float(len(models_for_asp))
    else:
        lg = models_for_asp(xb)

    prob = torch.sigmoid(lg)
    sc, idx = torch.topk(prob, k=k, dim=1)
    return sc.cpu().numpy(), idx.cpu().numpy()

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ASPECTS = ["BP","MF","CC"]

for asp in ASPECTS:
    if isinstance(models[asp], (list, tuple)):
        models[asp] = [m.to(device).eval() for m in models[asp]]
    else:
        models[asp] = models[asp].to(device).eval()

print({asp: type(models[asp]).__name__ for asp in ASPECTS})
if isinstance(models["BP"], list):
    print("BP ensemble size:", len(models["BP"]))

In [ ]:
import numpy as np
import torch

OUT_PATH = "/kaggle/working/submission_small.tsv"

TOPK_PER_ASPECT = {"BP": 100, "MF": 80, "CC": 80}
FINAL_CAP = 250 
MIN_SCORE = 0.20
TEST_BATCH = 2048

ASPECTS = ["BP","MF","CC"]

def fmt_3sig(x: float) -> str:
    if x <= 0: 
        return None
    if x > 1:
        x = 1.0
    return f"{x:.3g}"

@torch.no_grad()
def predict_topk(model, xb, k):
    prob = torch.sigmoid(model(xb))
    sc, idx = torch.topk(prob, k=k, dim=1)
    return sc.cpu().numpy(), idx.cpu().numpy()

written = 0
N = X_test.shape[0]

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for i0 in range(0, N, TEST_BATCH):
        xb_np = X_test[i0:i0+TEST_BATCH].astype(np.float32, copy=False)
        xb = torch.from_numpy(xb_np).to(device, non_blocking=True)

        batch_aspect = {}
        for asp in ASPECTS:
            k = min(TOPK_PER_ASPECT[asp], len(go_vocab[asp]))
            sc, idx = predict_topk_ensemble(models[asp], xb, k=k)  
            thr = float(thr_aspect.get(asp, MIN_SCORE)) if "thr_aspect" in globals() else MIN_SCORE
            batch_aspect[asp] = (sc, idx, thr)

        lines = []
        B = xb_np.shape[0]
        for r in range(B):
            pid = test_pids_aligned[i0 + r]
            combined = []

            for asp in ASPECTS:
                sc, idx, thr = batch_aspect[asp]
                for j in range(sc.shape[1]):
                    s = float(sc[r, j])
                    if s < thr or s < MIN_SCORE:
                        continue
                    go = go_vocab[asp][int(idx[r, j])]
                    combined.append((go, s))

            combined.sort(key=lambda x: x[1], reverse=True)
            combined = combined[:FINAL_CAP]

            for go, s in combined:
                s_txt = fmt_3sig(s)
                if s_txt is None:
                    continue
                lines.append(f"{pid}\t{go}\t{s_txt}\n")

        f.writelines(lines)
        written += len(lines)

print("Wrote:", OUT_PATH, "rows:", written)

In [ ]:
import os
print("MB:", os.path.getsize(OUT_PATH)/1024/1024)

In [ ]:
import pandas as pd, os

IN_PATH = "/kaggle/working/submission.tsv"
assert os.path.exists(IN_PATH)

sub = pd.read_csv(IN_PATH, sep="\t", header=None, names=["protein_id","go_id","score"])
print("rows:", len(sub))
print("proteins:", sub["protein_id"].nunique())
print("score min/max:", sub["score"].min(), sub["score"].max())